# Regression Models Overview

| | Method | Purpose |
|---|---|---|
| 1 | Linear Regression | Establish a simple linear relationship |
| 2 | Polynomial Regression | Capture nonlinear relationships |
| 3 | Ridge Regression | Reduce overfitting using L2 regularization |
| 4 | Lasso Regression | Feature selection using L1 regularization |
| 5 | ElasticNet Regression | Combine L1 and L2 regularization |

## Data Loading
We will load the preprocessed train, validation, and test datasets into pandas DataFrames.

In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/train_preprocessed.csv')
val_df = pd.read_csv('/content/validation_preprocessed.csv')
test_df = pd.read_csv('/content/test_preprocessed.csv')

display(train_df.head())

,trip_id,route_id,origin,destination,distance_km,route_type,departure_datetime,capacity,ticket_fare_inr,planned_event_flag,public_holiday_flag,passengers,journey_date,departure_hour,weekday,month,season,is_weekend,is_peak,ticket_fare_log
0,R11-20240101-05,R11,Patiala,Chandigarh,65.0,commuter,01-01-2024 05:00,40.0,125.53,0,0,12,2024-01-01,5.0,0.0,1,winter,0,0,4.840479
1,R01-20240101-17,R01,Amritsar,Chandigarh,230.0,intercity,01-01-2024 17:30,45.0,461.09,0,0,45,2024-01-01,17.0,0.0,1,winter,0,1,6.135760
2,R10-20240101-17,R10,Ludhiana,Amritsar,140.0,intercity,01-01-2024 17:30,45.0,264.40,0,0,30,2024-01-01,17.0,0.0,1,winter,0,1,5.581238
3,R09-20240101-17,R09,Delhi,Jaipur,280.0,intercity,01-01-2024 17:00,50.0,639.78,0,0,31,2024-01-01,17.0,0.0,1,winter,0,1,6.462686
4,R04-20240101-16,R04,Ludhiana,Chandigarh,100.0,intercity,01-01-2024 16:45,45.0,204.77,0,0,22,2024-01-01,16.0,0.0,1,winter,0,1,5.326759


## Feature Selection and Preprocessing
We will define our features and target variable. The target is `passengers`. We will select relevant features, perform one-hot encoding on categorical columns, and standardize numerical variables to ensure all models, especially regularized ones, function optimally.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

target = 'passengers'

feature_cols = [
    'distance_km', 'capacity', 'planned_event_flag',
    'public_holiday_flag', 'departure_hour', 'weekday',
    'month', 'is_weekend', 'is_peak', 'route_type', 'season'
]

X_train = train_df[feature_cols]
y_train = train_df[target]

X_val = val_df[feature_cols]
y_val = val_df[target]

X_test = test_df[feature_cols]
y_test = test_df[target]

num_cols = ['distance_km', 'capacity', 'departure_hour', 'weekday', 'month']
cat_cols = ['route_type', 'season']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

## Model Training and Evaluation
We will now train all five regression models:
1. Linear Regression
2. Polynomial Regression (Degree 2)
3. Ridge Regression
4. Lasso Regression
5. ElasticNet Regression

We will evaluate each model using Root Mean Squared Error (RMSE) and R-squared (R2) score on the validation dataset.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

results = {}

lr = LinearRegression()
lr.fit(X_train_proc, y_train)
lr_preds = lr.predict(X_val_proc)
results['Linear Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_val, lr_preds)),
    'R2': r2_score(y_val, lr_preds)
}

poly = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('linear_regression', LinearRegression())
])
poly.fit(X_train_proc, y_train)
poly_preds = poly.predict(X_val_proc)
results['Polynomial Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_val, poly_preds)),
    'R2': r2_score(y_val, poly_preds)
}

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_proc, y_train)
ridge_preds = ridge.predict(X_val_proc)
results['Ridge Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_val, ridge_preds)),
    'R2': r2_score(y_val, ridge_preds)
}

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_proc, y_train)
lasso_preds = lasso.predict(X_val_proc)
results['Lasso Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_val, lasso_preds)),
    'R2': r2_score(y_val, lasso_preds)
}

elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train_proc, y_train)
elastic_preds = elastic.predict(X_val_proc)
results['ElasticNet Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_val, elastic_preds)),
    'R2': r2_score(y_val, elastic_preds)
}

results_df = pd.DataFrame(results).T
display(results_df)

,RMSE,R2
Linear Regression,9.030538,0.420077
Polynomial Regression,7.946257,0.550977
Ridge Regression,9.030997,0.420018
Lasso Regression,9.061728,0.416064
ElasticNet Regression,9.095654,0.411684


## Detailed Model Evaluation

We will now evaluate all trained models on the validation dataset using the following specific metrics:
- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **WAPE** (Weighted Absolute Percentage Error)
- **R²** (Coefficient of Determination)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

def calculate_wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

models = {
    'Linear Regression': lr_preds,
    'Polynomial Regression': poly_preds,
    'Ridge Regression': ridge_preds,
    'Lasso Regression': lasso_preds,
    'ElasticNet Regression': elastic_preds
}

metric_results = {}

for name, preds in models.items():
    metric_results[name] = {
        'MAE': mean_absolute_error(y_val, preds),
        'RMSE': np.sqrt(mean_squared_error(y_val, preds)),
        'WAPE': calculate_wape(y_val, preds),
        'R²': r2_score(y_val, preds)
    }

metrics_df = pd.DataFrame(metric_results).T
display(metrics_df)

,MAE,RMSE,WAPE,R²
Linear Regression,7.154048,9.030538,0.226144,0.420077
Polynomial Regression,6.103811,7.946257,0.192945,0.550977
Ridge Regression,7.154658,9.030997,0.226163,0.420018
Lasso Regression,7.235608,9.061728,0.228722,0.416064
ElasticNet Regression,7.334140,9.095654,0.231836,0.411684


## Higher Degree Polynomial Regression

We will evaluate whether increasing the polynomial degree (Degree 3 and Degree 4) improves our validation metrics.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

def calculate_wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

high_poly_results = {}

for degree in [3, 4]:
    poly_pipeline = Pipeline([
        ('poly_features', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear_regression', LinearRegression())
    ])
    poly_pipeline.fit(X_train_proc, y_train)
    poly_preds = poly_pipeline.predict(X_val_proc)

    high_poly_results[f'Polynomial Regression (Degree {degree})'] = {
        'MAE': mean_absolute_error(y_val, poly_preds),
        'RMSE': np.sqrt(mean_squared_error(y_val, poly_preds)),
        'WAPE': calculate_wape(y_val, poly_preds),
        'R%': r2_score(y_val, poly_preds)
    }

high_poly_df = pd.DataFrame(high_poly_results).T
display(high_poly_df)

,MAE,RMSE,WAPE,R%
Polynomial Regression (Degree 3),6.031628,7.891935,0.190663,0.557096
Polynomial Regression (Degree 4),6.082267,7.971487,0.192264,0.548121


## Hyperparameter Tuning

We will now systematically tune the hyperparameters for our regression models on the validation split:
- **Polynomial Regression**: Tuning degree in `[1, 2, 3]`
- **Ridge Regression**: Tuning alpha in `[0.01, 0.1, 1.0, 10.0, 100.0]`
- **Lasso Regression**: Tuning alpha in `[0.001, 0.01, 0.1, 1.0, 10.0]`
- **ElasticNet Regression**: Tuning alpha in `[0.001, 0.01, 0.1, 1.0]` and l1_ratio in `[0.2, 0.5, 0.8]`

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

def calculate_wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

tuned_results = {}

best_poly_score = float('inf')
best_poly_degree = 2
best_poly_preds = None

for d in [1, 2, 3]:
    poly_pipe = Pipeline([
        ('poly_features', PolynomialFeatures(degree=d, include_bias=False)),
        ('linear_regression', LinearRegression())
    ])
    poly_pipe.fit(X_train_proc, y_train)
    preds = poly_pipe.predict(X_val_proc)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    if rmse < best_poly_score:
        best_poly_score = rmse
        best_poly_degree = d
        best_poly_preds = preds

tuned_results['Polynomial Regression'] = {
    'Best Params': f'degree={best_poly_degree}',
    'MAE': mean_absolute_error(y_val, best_poly_preds),
    'RMSE': best_poly_score,
    'WAPE': calculate_wape(y_val, best_poly_preds),
    'R%': r2_score(y_val, best_poly_preds)
}

best_ridge_score = float('inf')
best_ridge_alpha = 1.0
best_ridge_preds = None

for a in [0.01, 0.1, 1.0, 10.0, 100.0]:
    model = Ridge(alpha=a)
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_val_proc)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    if rmse < best_ridge_score:
        best_ridge_score = rmse
        best_ridge_alpha = a
        best_ridge_preds = preds

tuned_results['Ridge Regression'] = {
    'Best Params': f'alpha={best_ridge_alpha}',
    'MAE': mean_absolute_error(y_val, best_ridge_preds),
    'RMSE': best_ridge_score,
    'WAPE': calculate_wape(y_val, best_ridge_preds),
    'R%': r2_score(y_val, best_ridge_preds)
}

best_lasso_score = float('inf')
best_lasso_alpha = 0.1
best_lasso_preds = None

for a in [0.001, 0.01, 0.1, 1.0, 10.0]:
    model = Lasso(alpha=a, max_iter=2000)
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_val_proc)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    if rmse < best_lasso_score:
        best_lasso_score = rmse
        best_lasso_alpha = a
        best_lasso_preds = preds

tuned_results['Lasso Regression'] = {
    'Best Params': f'alpha={best_lasso_alpha}',
    'MAE': mean_absolute_error(y_val, best_lasso_preds),
    'RMSE': best_lasso_score,
    'WAPE': calculate_wape(y_val, best_lasso_preds),
    'R%': r2_score(y_val, best_lasso_preds)
}

best_elastic_score = float('inf')
best_elastic_alpha = 0.1
best_elastic_l1 = 0.5
best_elastic_preds = None

for a in [0.001, 0.01, 0.1, 1.0]:
    for l1 in [0.2, 0.5, 0.8]:
        model = ElasticNet(alpha=a, l1_ratio=l1, max_iter=2000)
        model.fit(X_train_proc, y_train)
        preds = model.predict(X_val_proc)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        if rmse < best_elastic_score:
            best_elastic_score = rmse
            best_elastic_alpha = a
            best_elastic_l1 = l1
            best_elastic_preds = preds

tuned_results['ElasticNet Regression'] = {
    'Best Params': f'alpha={best_elastic_alpha}, l1_ratio={best_elastic_l1}',
    'MAE': mean_absolute_error(y_val, best_elastic_preds),
    'RMSE': best_elastic_score,
    'WAPE': calculate_wape(y_val, best_elastic_preds),
    'R%': r2_score(y_val, best_elastic_preds)
}

tuned_df = pd.DataFrame(tuned_results).T
display(tuned_df)

,Best Params,MAE,RMSE,WAPE,R%
Polynomial Regression,degree=3,6.031628,7.891935,0.190663,0.557096
Ridge Regression,alpha=0.01,7.154054,9.030543,0.226144,0.420076
Lasso Regression,alpha=0.001,7.163361,9.039184,0.226438,0.418966
ElasticNet Regression,"alpha=0.001, l1_ratio=0.2",7.161033,9.035241,0.226364,0.419473


## Final Evaluation on Unseen Test Data

We evaluate our final selected models on the completely unseen test split to measure their generalization capabilities:
1. **Linear Regression**
2. **Tuned Polynomial Regression (Degree 3)**

In [ ]:

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# WAPE calculation
def calculate_wape(y_true, y_pred):
    return (
        np.sum(np.abs(y_true - y_pred))
        / np.sum(np.abs(y_true))
    ) * 100


# ---------------------------------------
# 1. Recreate models with best parameters
# ---------------------------------------

best_models = {}

# Linear Regression
best_models["Linear Regression"] = LinearRegression()

# Polynomial Regression
best_models["Polynomial Regression"] = Pipeline([
    ("poly_features", PolynomialFeatures(
        degree=best_poly_degree,
        include_bias=False
    )),
    ("linear_regression", LinearRegression())
])

# Ridge
best_models["Ridge Regression"] = Ridge(
    alpha=best_ridge_alpha
)

# Lasso
best_models["Lasso Regression"] = Lasso(
    alpha=best_lasso_alpha,
    max_iter=2000
)

# ElasticNet
best_models["ElasticNet Regression"] = ElasticNet(
    alpha=best_elastic_alpha,
    l1_ratio=best_elastic_l1,
    max_iter=2000
)


# ---------------------------------------
# 2. Train on training data
# ---------------------------------------

test_results = {}
test_predictions = {}

for name, model in best_models.items():

    # Fit only on training data
    model.fit(X_train_proc, y_train)

    # Predict unseen test data
    predictions = model.predict(X_test_proc)

    test_predictions[name] = predictions

    # Calculate metrics
    test_results[name] = {
        "MAE": mean_absolute_error(
            y_test, predictions
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_test, predictions
            )
        ),

        "WAPE (%)": calculate_wape(
            y_test, predictions
        ),

        "R2": r2_score(
            y_test, predictions
        )
    }


# ---------------------------------------
# 3. Display final test results
# ---------------------------------------

test_results_df = pd.DataFrame(
    test_results
).T

test_results_df = test_results_df.sort_values(
    by="RMSE"
)

print("FINAL UNSEEN TEST RESULTS")

display(test_results_df)


FINAL UNSEEN TEST RESULTS


,MAE,RMSE,WAPE (%),R2
Polynomial Regression,6.495983,8.465426,19.700313,0.470374
Linear Regression,7.347852,9.121283,22.283768,0.385129
Ridge Regression,7.347856,9.121286,22.283780,0.385129
ElasticNet Regression,7.353001,9.124362,22.299385,0.384714
Lasso Regression,7.352503,9.126257,22.297873,0.384458


Normal KNN model training

In [ ]:

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Initialize KNN
knn = KNeighborsRegressor()

# Train
knn.fit(X_train_proc, y_train)

# Validation prediction
knn_val_pred = knn.predict(X_val_proc)

# Evaluation
def evaluate_model(y_true, y_pred):

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "WAPE": (
            np.sum(np.abs(y_true - y_pred))
            / np.sum(np.abs(y_true))
        ) * 100,
        "R2": r2_score(y_true, y_pred)
    }

knn_results = evaluate_model(
    y_val, knn_val_pred
)

print("KNN Default Model Results:")
print(knn_results)


KNN Default Model Results:
{'MAE': 6.491568955890565, 'RMSE': np.float64(8.584050888027237), 'WAPE': np.float64(20.52022432743687), 'R2': 0.47600439171598574}


KNN hyperparameter tuning
We will experiment with:
Hyperparameter	Values
n_neighbors	3, 5, 7, 9, 11, 15, 20
weights	uniform, distance
p	1, 2

- n_neighbors: Number of nearest neighbors.
- weights: Equal weighting or closer neighbors receiving greater weight.
- p=1: Manhattan distance.
- p=2: Euclidean distance.

In [ ]:

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

knn_tuning_results = []

best_knn_score = float("inf")
best_knn_model = None
best_knn_params = None
best_knn_preds = None

for k in [3, 5, 7, 9, 11, 15, 20]:

    for weight in ["uniform", "distance"]:

        for p_value in [1, 2]:

            model = KNeighborsRegressor(
                n_neighbors=k,
                weights=weight,
                p=p_value
            )

            model.fit(X_train_proc, y_train)

            preds = model.predict(X_val_proc)

            rmse = np.sqrt(
                mean_squared_error(y_val, preds)
            )

            knn_tuning_results.append({
                "n_neighbors": k,
                "weights": weight,
                "p": p_value,
                "RMSE": rmse
            })

            if rmse < best_knn_score:

                best_knn_score = rmse
                best_knn_model = model
                best_knn_params = {
                    "n_neighbors": k,
                    "weights": weight,
                    "p": p_value
                }
                best_knn_preds = preds

knn_tuning_df = pd.DataFrame(
    knn_tuning_results
).sort_values("RMSE")

print("Best KNN Parameters:")
print(best_knn_params)

print("\nBest KNN Validation Results:")
print(evaluate_model(y_val, best_knn_preds))

display(knn_tuning_df)


Best KNN Parameters:
{'n_neighbors': 20, 'weights': 'uniform', 'p': 1}

Best KNN Validation Results:
{'MAE': 6.226305136795086, 'RMSE': np.float64(8.107429741640303), 'WAPE': np.float64(19.681710077526223), 'R2': 0.5325776574810301}


,n_neighbors,weights,p,RMSE
24,20,uniform,1,8.107430
25,20,uniform,2,8.132804
20,15,uniform,1,8.142568
21,15,uniform,2,8.161612
16,11,uniform,1,8.214713
17,11,uniform,2,8.229102
13,9,uniform,2,8.279049
12,9,uniform,1,8.284464
8,7,uniform,1,8.363145
9,7,uniform,2,8.375168


Support Vector Regression (SVR)
Normal SVR model training

In [ ]:

from sklearn.svm import SVR

# Initialize default SVR
svr = SVR(kernel="rbf")

# Train
svr.fit(X_train_proc, y_train)

# Validation prediction
svr_val_pred = svr.predict(X_val_proc)

# Evaluation
svr_results = evaluate_model(
    y_val, svr_val_pred
)

print("SVR Default Model Results:")
print(svr_results)


SVR Default Model Results:
{'MAE': 6.067984794729473, 'RMSE': np.float64(8.049287817294216), 'WAPE': np.float64(19.181250333994583), 'R2': 0.5392577982547726}


B. SVR hyperparameter tuning
Hyperparameter	Values
C	0.1, 1, 10, 100
gamma	scale, auto, 0.001, 0.01, 0.1
epsilon	0.1, 0.2, 0.5

In [ ]:

from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

svr_tuning_results = []

best_svr_score = float("inf")
best_svr_model = None
best_svr_params = None
best_svr_preds = None

for c_value in [0.1, 1, 10, 100]:

    for gamma_value in [
        "scale", "auto", 0.001, 0.01, 0.1
    ]:

        for epsilon_value in [0.1, 0.2, 0.5]:

            model = SVR(
                kernel="rbf",
                C=c_value,
                gamma=gamma_value,
                epsilon=epsilon_value
            )

            model.fit(X_train_proc, y_train)

            preds = model.predict(X_val_proc)

            rmse = np.sqrt(
                mean_squared_error(y_val, preds)
            )

            svr_tuning_results.append({
                "C": c_value,
                "gamma": gamma_value,
                "epsilon": epsilon_value,
                "RMSE": rmse
            })

            if rmse < best_svr_score:

                best_svr_score = rmse
                best_svr_model = model

                best_svr_params = {
                    "C": c_value,
                    "gamma": gamma_value,
                    "epsilon": epsilon_value
                }

                best_svr_preds = preds

svr_tuning_df = pd.DataFrame(
    svr_tuning_results
).sort_values("RMSE")

print("Best SVR Parameters:")
print(best_svr_params)

print("\nBest SVR Validation Results:")
print(evaluate_model(y_val, best_svr_preds))

display(svr_tuning_df)


Best SVR Parameters:
{'C': 100, 'gamma': 'auto', 'epsilon': 0.5}

Best SVR Validation Results:
{'MAE': 5.865248371872966, 'RMSE': np.float64(7.962467288370624), 'WAPE': np.float64(18.54038879426118), 'R2': 0.5491434304115012}


,C,gamma,epsilon,RMSE
50,100.0,auto,0.5,7.962467
44,10.0,0.1,0.5,7.965923
43,10.0,0.1,0.2,7.967512
32,10.0,scale,0.5,7.969100
42,10.0,0.1,0.1,7.970283
49,100.0,auto,0.2,7.974935
31,10.0,scale,0.2,7.975423
30,10.0,scale,0.1,7.975459
48,100.0,auto,0.1,7.978149
35,10.0,auto,0.5,7.990607
